<a href="https://colab.research.google.com/github/MithunSrinivas28/wafer-defect-ai/blob/main/Wafer_detect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Mount Google Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


### Load Dataset Using TensorFlow

In [2]:
TRAIN_PATH = "/content/drive/Shared-With-Me/Datasets/train"
TEST_PATH  = "/content/drive/Shared-With-Me/Datasets/test"


In [ ]:
import tensorflow as tf

TRAIN_PATH = "/content/drive/MyDrive/Datasets/train"
TEST_PATH  = "/content/drive/MyDrive/Datasets/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_data = tf.keras.preprocessing.image_dataset_from_directory(
    TEST_PATH,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [ ]:
import tensorflow as tf

# Reload only to get class names
temp_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/drive/MyDrive/Datasets/train",
    image_size=(224,224),
    batch_size=32
)

class_names = temp_ds.class_names
NUM_CLASSES = len(class_names)

print("Classes:", class_names)
print("Num classes:", NUM_CLASSES)


In [ ]:
import os

for c in os.listdir(TRAIN_PATH):
    print(c, len(os.listdir(TRAIN_PATH + "/" + c)))


### Normalize + Add Data Augmentation

In [ ]:
from tensorflow.keras import layers

# Normalize (0–255 → 0–1)
normalization = layers.Rescaling(1./255)

# Data Augmentation
#data_augmentation = tf.keras.Sequential([
 #   layers.RandomFlip("horizontal"),
   # layers.RandomRotation(0.2),
   # layers.RandomZoom(0.2),
   # layers.RandomContrast(0.2),
#])
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
])

# Apply to datasets
train_data = train_data.map(lambda x, y: (normalization(data_augmentation(x)), y))
test_data  = test_data.map(lambda x, y: (normalization(x), y))


### Build the MobileNet Model (Your AI Brain)

In [ ]:
from tensorflow.keras import layers, models
import tensorflow as tf

base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(224,224,3),
    include_top=False,
    #weights="imagenet"
    weights=None
  )

base_model.trainable = True

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    #layers.Dense(128, activation="relu"),
    #layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.summary()


##** Compile the Model**

In [ ]:
model.compile(
    #optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    #loss="sparse_categorical_crossentropy",
   optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

y = []
for _, labels in train_data:
    y.extend(labels.numpy())

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)

class_weights = dict(enumerate(class_weights))
print(class_weights)


### Train the Model

In [ ]:
EPOCHS = 15   # good for small dataset

history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=EPOCHS,
    class_weight=class_weights
)


In [ ]:
model.evaluate(train_data)
model.evaluate(test_data)
#model is overfitting